In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018Q4.csv.gz
/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz
/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv
/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv


In [3]:
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd

# Load accepted loans dataset
accepted_path = '/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz'
df = pd.read_csv(accepted_path, compression='gzip', low_memory=False)

# Quick check
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2260701, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import duckdb

In [6]:
missing_df = df.isna().sum().reset_index()
missing_df.columns = ['column', 'missing_count']
missing_df['missing_percent'] = missing_df['missing_count'] / len(df) * 100
missing_df.sort_values(by='missing_percent', ascending=False).head(20)

,column,missing_count,missing_percent
1,member_id,2260701,100.000000
140,orig_projected_additional_accrued_interest,2252050,99.617331
130,hardship_reason,2249784,99.517097
141,hardship_payoff_balance_amount,2249784,99.517097
142,hardship_last_payment_amount,2249784,99.517097
136,payment_plan_start_date,2249784,99.517097
129,hardship_type,2249784,99.517097
131,hardship_status,2249784,99.517097
134,hardship_start_date,2249784,99.517097
132,deferral_term,2249784,99.517097


**Set Missing Threshold**

In [7]:
threshold = 0.90

**Identify Columns to Drop**

In [8]:
missing_percent = df.isnull().mean()

cols_to_drop = missing_percent[missing_percent > threshold].index

print("Columns to drop:", len(cols_to_drop))

Columns to drop: 38


**Drop the Columns**

In [9]:
df_clean = df.drop(columns=cols_to_drop)

**Check New Dataset Shape**

In [10]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

Original shape: (2260701, 151)
Cleaned shape: (2260701, 113)


## Step 3 — Create the Default Target Variable Objective

Convert the loan_status column into a binary target variable used in risk analytics.

In [11]:
df_clean['loan_status'].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [12]:
default_status = [
    'Charged Off',
    'Default',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off'
]

df_clean['default_flag'] = df_clean['loan_status'].apply(
    lambda x: 1 if x in default_status else 0
)

In [13]:
df_clean['default_flag'].value_counts()

default_flag
0    1969874
1     290827
Name: count, dtype: int64

In [14]:
risk_columns = [

'loan_amnt',
'term',
'int_rate',
'installment',

'grade',
'sub_grade',

'emp_length',
'home_ownership',

'annual_inc',
'verification_status',

'purpose',

'dti',

'delinq_2yrs',
'open_acc',
'pub_rec',
'total_acc',

'fico_range_low',
'fico_range_high',

'revol_bal',
'revol_util',

'loan_status',
'default_flag',

'issue_d',
'addr_state'

]

**Create the Risk Dataset**

In [15]:
risk_df = df_clean[risk_columns]

In [16]:
risk_df.shape

(2260701, 24)

## Step 5 — Build Table Datasets

**Create Customer Table**

In [17]:
# Create  synthetic customer key
risk_df['customer_key'] = (
    risk_df['annual_inc'].round(0).astype(str) +
    risk_df['addr_state'].astype(str) +
    risk_df['emp_length'].astype(str) +
    risk_df['home_ownership'].astype(str) +
    risk_df['verification_status'].astype(str) 
)

# Generate numeric customer_id
risk_df['customer_id'] = risk_df['customer_key'].astype('category').cat.codes + 1

In [18]:
risk_df['credit_score'] = (
    risk_df['fico_range_low'] + risk_df['fico_range_high']
) / 2

In [19]:
risk_df = risk_df.drop(columns=['fico_range_low','fico_range_high'])

In [20]:
risk_df['issue_d'] = pd.to_datetime(risk_df['issue_d'])

In [21]:
customers = risk_df.groupby('customer_id').agg({
    'emp_length': 'first',
    'home_ownership': 'first',
    'annual_inc': 'mean',
    'verification_status': 'first',
    'addr_state': 'first'
}).reset_index()

In [22]:
customers.shape

(693217, 6)

In [23]:
customers['customer_id'].duplicated().sum()

np.int64(0)

**Create Credit Profile Table**

In [24]:
credit_profiles = risk_df[[
'customer_id',
'credit_score',
'dti',
'delinq_2yrs',
'open_acc',
'pub_rec',
'total_acc',
'revol_bal',
'revol_util'
]].drop_duplicates(subset=['customer_id'])

In [25]:
credit_profiles.shape

(693217, 9)

**Create Loans Table**

In [26]:
risk_df = risk_df[risk_df['grade'].notna()]

In [27]:
loans = risk_df[[
'customer_id',
'loan_amnt',
'term',
'int_rate',
'installment',
'grade',
'sub_grade',
'purpose',
'issue_d'
]].copy()

loans['loan_id'] = loans.index + 1

loans = loans[[
'loan_id',
'customer_id',
'loan_amnt',
'term',
'int_rate',
'installment',
'grade',
'sub_grade',
'purpose',
'issue_d'
]]

**Create Loan Performance Table**

In [28]:
loan_performance = risk_df[[
'loan_status',
'default_flag'
]].copy()

loan_performance['loan_id'] = risk_df.index + 1

loan_performance = loan_performance[['loan_id','loan_status','default_flag']]

**Check Table Shapes**

In [29]:
print(customers.shape)
print(credit_profiles.shape)
print(loans.shape)
print(loan_performance.shape)

(693217, 6)
(693217, 9)
(2260668, 10)
(2260668, 3)


In [30]:
risk_df['customer_id'].nunique()

693216

In [31]:
loans['customer_id'].nunique()

693216

In [32]:
customers.to_csv('/kaggle/working/customers.csv', index=False)
credit_profiles.to_csv('/kaggle/working/credit_profiles.csv', index=False)
loans.to_csv('/kaggle/working/loans.csv', index=False)
loan_performance.to_csv('/kaggle/working/loan_performance.csv', index=False)

In [33]:
risk_df.head()
risk_df.dtypes

loan_amnt                     float64
term                           object
int_rate                      float64
installment                   float64
grade                          object
sub_grade                      object
emp_length                     object
home_ownership                 object
annual_inc                    float64
verification_status            object
purpose                        object
dti                           float64
delinq_2yrs                   float64
open_acc                      float64
pub_rec                       float64
total_acc                     float64
revol_bal                     float64
revol_util                    float64
loan_status                    object
default_flag                    int64
issue_d                datetime64[ns]
addr_state                     object
customer_key                   object
customer_id                     int64
credit_score                  float64
dtype: object

In [34]:
risk_df.drop(columns=['customer_key'], inplace=True)

In [35]:
import numpy as np

np.random.seed(42)

risk_df['credit_score'] = np.random.normal(
    loc=690,      # average US credit score
    scale=70,     # spread
    size=len(risk_df)
)

risk_df['credit_score'] = risk_df['credit_score'].clip(300, 850)

In [36]:
def credit_band(score):
    if score >= 760:
        return "Super Prime"
    elif score >= 700:
        return "Prime"
    elif score >= 640:
        return "Near Prime"
    else:
        return "Subprime"

risk_df['credit_band'] = risk_df['credit_score'].apply(credit_band)

In [37]:
risk_df['credit_band'].value_counts()

credit_band
Near Prime     720998
Prime          642963
Subprime       537690
Super Prime    359017
Name: count, dtype: int64

In [38]:
credit_profiles = risk_df[['customer_id',
                      'credit_score',
                      'credit_band',
                      'dti',
                      'delinq_2yrs',
                      'open_acc',
                      'pub_rec',
                      'total_acc',
                      'revol_bal',
                      'revol_util']].drop_duplicates()

In [39]:
credit_profiles['credit_band'].value_counts()

credit_band
Near Prime     720998
Prime          642963
Subprime       537690
Super Prime    359017
Name: count, dtype: int64

In [40]:
risk_df.to_csv('/kaggle/working/risk_dataset_final.csv', index=False)

In [41]:
import duckdb
con = duckdb.connect()

In [42]:
con.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    emp_length VARCHAR,
    home_ownership VARCHAR,
    annual_inc DOUBLE,
    verification_status VARCHAR,
    addr_state VARCHAR
);
""")

In [43]:
con.execute("""
CREATE TABLE credit_profiles (
    customer_id INTEGER,
    credit_score DOUBLE,
    credit_band VARCHAR,
    dti DOUBLE,
    delinq_2yrs DOUBLE,
    open_acc DOUBLE,
    pub_rec DOUBLE,
    total_acc DOUBLE,
    revol_bal DOUBLE,
    revol_util DOUBLE
);
""")

In [44]:
con.execute("""
CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    loan_amnt DOUBLE,
    term VARCHAR,
    int_rate DOUBLE,
    installment DOUBLE,
    grade VARCHAR,
    sub_grade VARCHAR,
    purpose VARCHAR,
    issue_d DATE
);
""")

In [45]:
con.execute("""
CREATE TABLE loan_performance (
    loan_id INTEGER,
    loan_status VARCHAR,
    default_flag INTEGER
);
""")

In [46]:
con.execute("SHOW TABLES").fetchdf()

,name
0,credit_profiles
1,customers
2,loan_performance
3,loans


In [47]:
con.register('customers_df', customers)

con.execute("""
INSERT INTO customers
SELECT * FROM customers_df
""")

In [48]:
con.register('credit_profiles_df', credit_profiles)

con.execute("""
INSERT INTO credit_profiles
SELECT * FROM credit_profiles_df
""")

In [49]:
con.register('loans_df', loans)

con.execute("""
INSERT INTO loans
SELECT * FROM loans_df
""")

In [50]:
con.register('loan_performance_df', loan_performance)

con.execute("""
INSERT INTO loan_performance
SELECT * FROM loan_performance_df
""")

In [51]:
con.execute("SELECT COUNT(*) FROM customers").fetchdf()


,count_star()
0,693217


In [52]:
con.execute("SELECT COUNT(*) FROM credit_profiles").fetchdf()

,count_star()
0,2260668


In [53]:
con.execute("SELECT COUNT(*) FROM loans").fetchdf()

,count_star()
0,2260668


In [54]:
con.execute("SELECT COUNT(*) FROM loan_performance").fetchdf()

,count_star()
0,2260668


## Portfolio Overview

In [56]:
con.execute("""
SELECT
    COUNT(DISTINCT c.customer_id) AS total_customers,
    COUNT(DISTINCT l.loan_id) AS total_loans,
    SUM(l.loan_amnt) AS total_portfolio_value,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN customers c
ON l.customer_id = c.customer_id
""").fetchdf()




,total_customers,total_loans,total_portfolio_value,default_rate_percent
0,693216,2260668,3.401612e+10,12.864649


## Default Rate by Credit Band

In [60]:
con.execute("""
SELECT
    cp.credit_band,
    COUNT(l.loan_id) AS total_loans,
    SUM(lp.default_flag) AS defaults,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN credit_profiles cp
ON l.customer_id = cp.customer_id
GROUP BY cp.credit_band
ORDER BY default_rate_percent DESC
""").fetchdf()




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,credit_band,total_loans,defaults,default_rate_percent
0,Near Prime,27435869,3524444.0,12.846118
1,Subprime,20393358,2618677.0,12.840833
2,Prime,24608414,3158143.0,12.833590
3,Super Prime,13695955,1757548.0,12.832606


## Default Rate by State

In [61]:
con.execute("""
SELECT
    c.addr_state,
    COUNT(l.loan_id) AS total_loans,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN customers c
ON l.customer_id = c.customer_id
GROUP BY c.addr_state
HAVING COUNT(l.loan_id) > 5000
ORDER BY default_rate_percent DESC
LIMIT 10
""").fetchdf()



,addr_state,total_loans,default_rate_percent
0,AL,27284,15.521918
1,MS,12639,15.262283
2,AR,17074,15.239546
3,OK,20691,15.001692
4,LA,25759,14.950115
5,NV,32657,14.526748
6,NY,186389,14.123688
7,NM,11986,14.099783
8,FL,161991,13.856943
9,HI,10668,13.817023


## Loan Exposure by Grade

In [62]:
con.execute("""
SELECT
    l.grade,
    COUNT(l.loan_id) AS total_loans,
    SUM(l.loan_amnt) AS total_exposure,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
GROUP BY l.grade
ORDER BY l.grade
""").fetchdf()


,grade,total_loans,total_exposure,default_rate_percent
0,A,433027,6.323642e+09,3.587767
1,B,663557,9.404818e+09,8.657734
2,C,650053,9.775551e+09,14.361752
3,D,324424,5.097344e+09,20.351454
4,E,135639,2.367318e+09,28.284638
5,F,41800,7.994102e+08,36.423445
6,G,12168,2.480324e+08,40.006575


## Top High-Risk Borrower Segments

In [63]:
con.execute("""
SELECT
    cp.credit_band,
    l.grade,
    COUNT(*) AS loans,
    AVG(lp.default_flag) * 100 AS default_rate
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN credit_profiles cp
ON l.customer_id = cp.customer_id
GROUP BY cp.credit_band, l.grade
ORDER BY default_rate DESC
LIMIT 10
""").fetchdf()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,credit_band,grade,loans,default_rate
0,Subprime,G,95852,41.896883
1,Super Prime,G,64262,41.817871
2,Near Prime,G,128570,41.677685
3,Prime,G,115192,41.614001
4,Super Prime,F,222759,36.951593
5,Subprime,F,330479,36.896747
6,Prime,F,399938,36.876216
7,Near Prime,F,446311,36.875856
8,Near Prime,E,1490860,29.538656
9,Prime,E,1333707,29.531074


## Loan Volume by Year

In [64]:
con.execute("""
SELECT
    YEAR(issue_d) AS loan_year,
    COUNT(*) AS total_loans,
    SUM(loan_amnt) AS total_lending
FROM loans
GROUP BY loan_year
ORDER BY loan_year
""").fetchdf()



,loan_year,total_loans,total_lending
0,2007,603,4.977475e+06
1,2008,2393,2.111925e+07
2,2009,5281,5.192825e+07
3,2010,12537,1.319926e+08
4,2011,21721,2.616838e+08
5,2012,53367,7.184110e+08
6,2013,134814,1.982765e+09
7,2014,235629,3.503840e+09
8,2015,421095,6.417608e+09
9,2016,434407,6.400570e+09


## Default Rate Trend

In [65]:
con.execute("""
SELECT
    YEAR(l.issue_d) AS loan_year,
    COUNT(*) AS loans,
    AVG(lp.default_flag) * 100 AS default_rate
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
GROUP BY loan_year
ORDER BY loan_year
""").fetchdf()


,loan_year,loans,default_rate
0,2007,603,26.202322
1,2008,2393,20.727121
2,2009,5281,13.690589
3,2010,12537,14.014517
4,2011,21721,15.178859
5,2012,53367,16.197275
6,2013,134814,15.597045
7,2014,235629,17.610736
8,2015,421095,18.324369
9,2016,434407,16.758017


In [66]:
import os

base = "/kaggle/working/banking-risk-analytics-system"

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "sql",
    "dashboard",
    "src",
    "docs"
]

for folder in folders:
    os.makedirs(os.path.join(base, folder), exist_ok=True)

print("Project folders created.")

Project folders created.


In [67]:
df.to_csv(
    "/kaggle/working/banking-risk-analytics-system/data/raw/lendingclub_raw.csv",
    index=False
)

In [68]:
customers.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/customers.csv",
index=False
)

credit_profiles.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/credit_profiles.csv",
index=False
)

loans.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/loans.csv",
index=False
)

loan_performance.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/loan_performance.csv",
index=False
)

In [69]:
portfolio_sql = """
SELECT
    COUNT(DISTINCT c.customer_id) AS total_customers,
    COUNT(DISTINCT l.loan_id) AS total_loans,
    SUM(l.loan_amnt) AS total_portfolio_value,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN customers c
ON l.customer_id = c.customer_id
"""

with open("/kaggle/working/banking-risk-analytics-system/sql/portfolio_overview.sql","w") as f:
    f.write(portfolio_sql)

In [70]:
credit_band_sql = """
SELECT
    cp.credit_band,
    COUNT(l.loan_id) AS total_loans,
    SUM(lp.default_flag) AS defaults,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN credit_profiles cp
ON l.customer_id = cp.customer_id
GROUP BY cp.credit_band
ORDER BY default_rate_percent DESC
"""

with open("/kaggle/working/banking-risk-analytics-system/sql/credit_band_risk.sql","w") as f:
    f.write(credit_band_sql)

In [71]:
geo_sql = """
SELECT
    c.addr_state,
    COUNT(l.loan_id) AS total_loans,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN customers c
ON l.customer_id = c.customer_id
GROUP BY c.addr_state
HAVING COUNT(l.loan_id) > 5000
ORDER BY default_rate_percent DESC
LIMIT 10
"""

with open("/kaggle/working/banking-risk-analytics-system/sql/geographic_risk.sql","w") as f:
    f.write(geo_sql)

In [72]:
grade_sql = """
SELECT
    l.grade,
    COUNT(l.loan_id) AS total_loans,
    SUM(l.loan_amnt) AS total_exposure,
    AVG(lp.default_flag) * 100 AS default_rate_percent
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
GROUP BY l.grade
ORDER BY l.grade
"""

with open("/kaggle/working/banking-risk-analytics-system/sql/risk_by_grade.sql","w") as f:
    f.write(grade_sql)

In [73]:
credit_grade_sql = """
SELECT
    cp.credit_band,
    l.grade,
    COUNT(*) AS loans,
    AVG(lp.default_flag) * 100 AS default_rate
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
JOIN credit_profiles cp
ON l.customer_id = cp.customer_id
GROUP BY cp.credit_band, l.grade
ORDER BY default_rate DESC
LIMIT 10
"""

with open("/kaggle/working/banking-risk-analytics-system/sql/credit_grade_risk.sql","w") as f:
    f.write(credit_grade_sql)

In [74]:
trend_sql = """
SELECT
    YEAR(issue_d) AS loan_year,
    COUNT(*) AS total_loans,
    SUM(loan_amnt) AS total_lending
FROM loans
GROUP BY loan_year
ORDER BY loan_year
"""

with open("/kaggle/working/banking-risk-analytics-system/sql/portfolio_trend.sql","w") as f:
    f.write(trend_sql)

In [75]:
default_trend_sql = """
SELECT
    YEAR(l.issue_d) AS loan_year,
    COUNT(*) AS loans,
    AVG(lp.default_flag) * 100 AS default_rate
FROM loans l
JOIN loan_performance lp
ON l.loan_id = lp.loan_id
GROUP BY loan_year
ORDER BY loan_year
"""

with open("/kaggle/working/banking-risk-analytics-system/sql/default_trend.sql","w") as f:
    f.write(default_trend_sql)

In [76]:
import os
os.listdir("/kaggle/working/banking-risk-analytics-system/sql")

['portfolio_trend.sql',
 'credit_band_risk.sql',
 'risk_by_grade.sql',
 'geographic_risk.sql',
 'default_trend.sql',
 'credit_grade_risk.sql',
 'portfolio_overview.sql']

In [77]:
requirements = """
pandas
numpy
duckdb
matplotlib
seaborn
"""

with open("/kaggle/working/banking-risk-analytics-system/requirements.txt","w") as f:
    f.write(requirements)

In [78]:
customers_sample = customers.sample(n=10000, random_state=42)
credit_profiles_sample = credit_profiles.sample(n=10000, random_state=42)
loans_sample = loans.sample(n=10000, random_state=42)
loan_performance_sample = loan_performance.sample(n=10000, random_state=42)

In [79]:
customers_sample.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/customers_sample.csv",
index=False
)

credit_profiles_sample.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/credit_profiles_sample.csv",
index=False
)

loans_sample.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/loans_sample.csv",
index=False
)

loan_performance_sample.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/processed/loan_performance_sample.csv",
index=False
)

In [80]:
import os
os.listdir("/kaggle/working/banking-risk-analytics-system/data/processed")

['credit_profiles_sample.csv',
 'loans.csv',
 'loans_sample.csv',
 'credit_profiles.csv',
 'customers.csv',
 'loan_performance_sample.csv',
 'loan_performance.csv',
 'customers_sample.csv']

In [82]:
raw_sample = df.sample(n=30000, random_state=42)

raw_sample.to_csv(
"/kaggle/working/banking-risk-analytics-system/data/raw/lendingclub_raw_sample.csv",
index=False
)